In [63]:
# Import required libraries and setup
import os
import json
import time
import random
import numpy as np
import requests
import pandas as pd

In [ ]:
# Set your API configuration
LLM_BASE_URL = ""
LLM_API_KEY = ""

print("LLM Client initialized successfully!")

LLM Client initialized successfully!


In [65]:
# Check available models and update MODELS list
def check_available_models():
    """Check what models are available in your Ollama instance"""
    try:
        response = requests.get(f"{LLM_BASE_URL}/api/tags")
        if response.status_code == 200:
            data = response.json()
            available_models = [model['name'] for model in data.get('models', [])]
            print("Available models:")
            for model in available_models:
                print(f"  - {model}")
            return available_models
        else:
            print(f"Error checking models: HTTP {response.status_code}")
            return []
    except Exception as e:
        print(f"Error checking models: {e}")
        # Fallback to common model names
        fallback_models = ["llama3.1:latest", "llama3:latest", "mistral:latest", "qwen2.5:latest"]
        print(f"Using fallback models: {fallback_models}")
        return fallback_models

# Check available models and update MODELS list
available_models = check_available_models()

Available models:
  - gpt-oss:latest
  - qwen3:latest
  - llama3.1:latest
  - mistral:latest
  - qwen2.5:latest
  - gemma3:latest
  - deepseek-r1:latest
  - codellama:latest
  - deepseek-coder-v2:latest
  - qwen2.5-coder:32b
  - deepseek-r1:32b


In [66]:
# Define the models and review parameters
MODELS = ["mistral:latest", "llama3.1:latest", "codellama:latest","qwen2.5-coder:32b"]  
NUM_REVIEWS_PER_RESTAURANT_PER_MODEL = 10  
SAMPLE_SIZE = 50  # Number of restaurants to sample
RANDOM_SEED = 42   # Fixed seed for reproducibility

In [67]:
# Load the dataset
prompt_df = pd.read_csv("prompt_bank.csv")
prompt_df = prompt_df[['review', 'restaurant_name', 'rating_value', 'category', 'keywords']]

In [68]:
# Sample restaurants with fixed seed for reproducibility
sampled_restaurants = prompt_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
print(f"Sampled {len(sampled_restaurants)} restaurants with seed {RANDOM_SEED}")

Sampled 50 restaurants with seed 42


In [69]:
prompt_template = """Write a realistic Google restaurant review for {restaurant_name} that serves {category} food.

Guidelines:
- Write a natural, authentic review like a real customer would write
- Length should vary naturally: some short (1-2 sentences), some medium (3-4 sentences), some longer (4-6 sentences)
- Sound like a real person, not an AI
- Mention specific details about food quality, service, or atmosphere
- Use the keywords naturally: {keywords}
- Make it personal and authentic
- Do NOT include any ratings, star ratings or numerical scores in the review text

IMPORTANT: Output ONLY the review text, no thinking process, no explanations, no XML tags.

Review:"""

print(f"Will generate {NUM_REVIEWS_PER_RESTAURANT_PER_MODEL} reviews for each of {len(sampled_restaurants)} restaurants")
print(f"Total: {NUM_REVIEWS_PER_RESTAURANT_PER_MODEL * len(sampled_restaurants)} reviews per model")
print(f"Models: {MODELS}")
print(f"Sample seed: {RANDOM_SEED}")

Will generate 10 reviews for each of 50 restaurants
Total: 500 reviews per model
Models: ['mistral:latest', 'llama3.1:latest', 'codellama:latest', 'qwen2.5-coder:32b']
Sample seed: 42


In [70]:
# Define the review generation function using requests
def generate_review(model, restaurant_name, category, keywords):
    """Generate a single review using the specified model via HTTP"""

    prompt = prompt_template.format(
        restaurant_name=restaurant_name,
        category=category,
        keywords=keywords
    )

    try:

        # Vary temperature slightly to encourage different writing styles
        temp_variation = random.uniform(0.6, 0.9)

        payload = {
            "model": model,
            "prompt": prompt,
            "options": {
                "temperature": temp_variation,
                "num_predict": random.choice([80, 120, 150, 200])
            },
            "stream": False
        }

        response = requests.post(f"{LLM_BASE_URL}/api/generate", json=payload)

        if response.status_code == 200:
            data = response.json()
            review_text = data.get('response', '').strip()
            return review_text
        else:
            print(f"HTTP Error {response.status_code} for model {model}: {response.text}")
            return None

    except Exception as e:
        print(f"Error generating review with {model}: {e}")
        return None

In [71]:
# === Main loop: generate for all models ===
def generate_all_reviews():
    all_reviews = {}
    
    # Set random seed for consistent rating variations
    random.seed(RANDOM_SEED)

    for model in MODELS:
        print(f"\nGenerating reviews for {model}...")
        model_reviews = []
        model_success_count = 0

        for _, restaurant in sampled_restaurants.iterrows():
            print(f"  Restaurant: {restaurant['restaurant_name']}")
            restaurant_reviews = []
            restaurant_success_count = 0

            for i in range(NUM_REVIEWS_PER_RESTAURANT_PER_MODEL):
                # Generate a random rating between 1.0 and 5.0 for the AI review
                ai_rating = round(random.uniform(1.0, 5.0), 1)

                keywords = restaurant['keywords']
                if isinstance(keywords, list):
                    keywords = ', '.join(keywords)
                elif pd.isna(keywords):
                    keywords = "food, service"

                review = generate_review(
                    model=model,
                    restaurant_name=restaurant['restaurant_name'],
                    category=restaurant['category'],
                    keywords=keywords
                )

                if review:
                    review_data = {
                        'model': model,
                        'restaurant': restaurant['restaurant_name'],
                        'category': restaurant['category'],
                        'ai_rating': ai_rating,  # Randomly generated rating for AI review
                        'human_rating': restaurant['rating_value'],  # Original human rating
                        'keywords': keywords,
                        'ai_review': review,  # AI-generated review text
                        'human_review': restaurant['review'],  # Original human review
                        'word_count': len(review.split())
                    }
                    model_reviews.append(review_data)
                    restaurant_reviews.append(review_data)
                    restaurant_success_count += 1
                    model_success_count += 1
                else:
                    print(f"    Failed to generate review {i+1} for {restaurant['restaurant_name']}")

                time.sleep(0.1)

            print(f"    ✅ Generated {restaurant_success_count}/{NUM_REVIEWS_PER_RESTAURANT_PER_MODEL} reviews for {restaurant['restaurant_name']}")

        all_reviews[model] = model_reviews
        total_expected = NUM_REVIEWS_PER_RESTAURANT_PER_MODEL * len(sampled_restaurants)
        print(f"✅ Completed {model}: {model_success_count}/{total_expected} reviews generated")

    return all_reviews

# === Run generation ===
print("Starting review generation...")
print(f"Using random seed: {RANDOM_SEED}")
print(f"Sample size: {SAMPLE_SIZE} restaurants")
all_reviews = generate_all_reviews()

Starting review generation...
Using random seed: 42
Sample size: 50 restaurants

Generating reviews for mistral:latest...
  Restaurant: eat.fit
    ✅ Generated 10/10 reviews for eat.fit
  Restaurant: PourHouse7
    ✅ Generated 10/10 reviews for PourHouse7
  Restaurant: eat.fit
    ✅ Generated 10/10 reviews for eat.fit
  Restaurant: The Fisherman's Wharf
    ✅ Generated 10/10 reviews for The Fisherman's Wharf
  Restaurant: Paradise
    ✅ Generated 10/10 reviews for Paradise
  Restaurant: The Glass Onion
    ✅ Generated 10/10 reviews for The Glass Onion
  Restaurant: Labonel
    ✅ Generated 10/10 reviews for Labonel
  Restaurant: Sardarji's Chaats & More
    ✅ Generated 10/10 reviews for Sardarji's Chaats & More
  Restaurant: Banana Leaf Multicuisine Restaurant
    ✅ Generated 10/10 reviews for Banana Leaf Multicuisine Restaurant
  Restaurant: Owm Nom Nom
    ✅ Generated 10/10 reviews for Owm Nom Nom
  Restaurant: Frio Bistro
    ✅ Generated 10/10 reviews for Frio Bistro
  Restaurant: La

In [72]:
# Save reviews for each model into separate JSON files
def save_reviews_by_model(all_reviews):
    """Save reviews for each model into separate JSON files."""
    for model, reviews in all_reviews.items():
        # Create a safe filename using the model name
        safe_model_name = model.replace(':', '_').replace('/', '_')
        filename = f"{safe_model_name}_reviews.json"

        # Prepare the data to save
        reviews_data = {
            "model": model,
            "total_reviews": len(reviews),
            "reviews": reviews
        }

        # Save the data to a JSON file
        try:
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(reviews_data, f, indent=4, ensure_ascii=False)
            print(f"✅ Reviews for model '{model}' saved to {filename}")
        except Exception as e:
            print(f"❌ Failed to save reviews for model '{model}': {e}")

# Call the function to save reviews
save_reviews_by_model(all_reviews)

✅ Reviews for model 'mistral:latest' saved to mistral_latest_reviews.json
✅ Reviews for model 'llama3.1:latest' saved to llama3.1_latest_reviews.json
✅ Reviews for model 'codellama:latest' saved to codellama_latest_reviews.json
✅ Reviews for model 'qwen2.5-coder:32b' saved to qwen2.5-coder_32b_reviews.json


In [73]:
# Save reviews to files and display statistics
def generate_statistics(all_reviews):
    """Generate statistics for the generated reviews"""
    statistics = {}
    
    for model, reviews in all_reviews.items():
        if not reviews:
            continue
            
        # Calculate statistics
        word_counts = [review['word_count'] for review in reviews]
        ai_ratings = [review['ai_rating'] for review in reviews]
        human_ratings = [review['human_rating'] for review in reviews]
        
        stats = {
            'total_reviews': len(reviews),
            'avg_word_count': sum(word_counts) / len(word_counts),
            'min_word_count': min(word_counts),
            'max_word_count': max(word_counts),
            'avg_ai_rating': sum(ai_ratings) / len(ai_ratings),
            'avg_human_rating': sum(human_ratings) / len(human_ratings),
            'ai_rating_distribution': {
                '1.0': len([r for r in ai_ratings if r == 1.0]),
                '2.0': len([r for r in ai_ratings if r == 2.0]),
                '3.0': len([r for r in ai_ratings if r == 3.0]),
                '4.0': len([r for r in ai_ratings if r == 4.0]),
                '5.0': len([r for r in ai_ratings if r == 5.0])
            }
        }
        
        statistics[model] = stats
        
        print(f"\n📊 Statistics for {model}:")
        print(f"   Total reviews: {stats['total_reviews']}")
        print(f"   Word count: {stats['min_word_count']}-{stats['max_word_count']} (avg: {stats['avg_word_count']:.1f})")
        print(f"   Average AI rating: {stats['avg_ai_rating']:.1f} stars")
        print(f"   Average human rating: {stats['avg_human_rating']:.1f} stars")
        print(f"   AI rating distribution: {stats['ai_rating_distribution']}")
    
    return statistics

# Generate and display statistics
print("\n" + "="*50)
print("REVIEW GENERATION STATISTICS")
print("="*50)
stats = generate_statistics(all_reviews)


REVIEW GENERATION STATISTICS

📊 Statistics for mistral:latest:
   Total reviews: 500
   Word count: 50-153 (avg: 90.4)
   Average AI rating: 3.0 stars
   Average human rating: 3.1 stars
   AI rating distribution: {'1.0': 3, '2.0': 15, '3.0': 11, '4.0': 11, '5.0': 3}

📊 Statistics for llama3.1:latest:
   Total reviews: 500
   Word count: 57-179 (avg: 112.0)
   Average AI rating: 2.9 stars
   Average human rating: 3.1 stars
   AI rating distribution: {'1.0': 5, '2.0': 15, '3.0': 19, '4.0': 12, '5.0': 7}

📊 Statistics for codellama:latest:
   Total reviews: 500
   Word count: 47-154 (avg: 86.7)
   Average AI rating: 3.0 stars
   Average human rating: 3.1 stars
   AI rating distribution: {'1.0': 10, '2.0': 10, '3.0': 11, '4.0': 9, '5.0': 2}

📊 Statistics for qwen2.5-coder:32b:
   Total reviews: 500
   Word count: 57-172 (avg: 88.5)
   Average AI rating: 3.0 stars
   Average human rating: 3.1 stars
   AI rating distribution: {'1.0': 7, '2.0': 12, '3.0': 10, '4.0': 11, '5.0': 3}
